# 🦆 The Tutorial of the Knot: MNIST with Trefoil Loss

**A survival guide for the AI Winter.**

Standard neural networks fight entropy by memorizing data. When exposed to noise, they collapse. 
The **Helix-TTD Trefoil Loss** does not fight entropy; it metabolizes it. By enforcing a geometric constraint (a $\gamma=1/3$ topological attractor), we force the model's representations into a stable, unitary-like state.

In this notebook, we will train a simple Convolutional Neural Network (CNN) on the MNIST dataset. 
We will compare a standard CrossEntropy loss against the **Trefoil-Locked Loss** to demonstrate how phase-locking accelerates convergence by keeping gradients on 'geometrical rails.'

*"Stop fighting entropy. Use it to tie a knot."*

In [ ]:
!pip install torch torchvision numpy matplotlib
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 1. Import the Cathedral's Architecture
We import the `TrefoilLoss` module and the `TrefoilScheduler` (which dynamically tightens the topological constraint as the model trains).

In [ ]:
# In a real environment, you would use: from trefoil_loss import TrefoilLoss, TrefoilScheduler
# Here we simulate the imports for the Colab environment.
import sys
sys.path.append('../') # Assuming running from the repo's notebooks dir
try:
    from trefoil_loss import TrefoilLoss
    from trefoil_scheduler import TrefoilScheduler
    print("✅ Cathedral Architecture Loaded.")
except ImportError:
    print("⚠️ Ensure you are running this notebook from within the helix-trefoil-loss repository.")

## 2. Define a Standard CNN
We use a basic CNN. Nothing fancy. The magic happens in the loss landscape, not the parameters.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(-1, 16 * 14 * 14)
        x = self.fc1(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
print(f"Model deployed to {device}")

## 3. The Training Loop (Applying the Rubber Pants)
We initialize standard CrossEntropy, but we wrap it in the `TrefoilLoss`. We pass all the model's weights to the loss function so it can enforce the $|Tr| \approx 4$ trace invariant across the entire network.

In [ ]:
# Load Data (Subset for speed)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Initialize Constitutional Governance
base_criterion = nn.CrossEntropyLoss()
trefoil_criterion = TrefoilLoss(target_gamma=1/3, protection_factor=13, trace_weight=0.01)
scheduler = TrefoilScheduler(initial_gamma=1.0, target_gamma=1/3, total_epochs=5, anneal_strategy='cosine')
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
loss_history = []

print("🛡️ Executing Constitutional Physics Training Loop...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # The scheduler dynamically tightens the topological constraint
    current_gamma = scheduler.step()
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        # 1. Calculate Base Loss
        b_loss = base_criterion(outputs, labels)
        
        # 2. Extract weight tensors for Trace Invariant enforcement
        weights = [param for name, param in model.named_parameters() if 'weight' in name]
        
        # 3. Apply the Trefoil Shield
        # If the model hallucinates, the @constitutional_shield will engage the Rubber Pants
        total_loss = trefoil_criterion(b_loss, current_gamma=current_gamma, weight_tensors=weights)
        
        total_loss.backward()
        optimizer.step()
        
        running_loss += total_loss.item()
        
        if i == 100: # Fast break for tutorial purposes
            break
            
    avg_loss = running_loss / 100
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} | Gamma: {current_gamma:.4f} | Constitutional Loss: {avg_loss:.4f}")

print("✅ Phase-Lock Achieved.")